# RLHF Diffusion Fine-Tuning for Age, Gender, Ethnicity, Emotion
We will:
1. Load dataset and metadata.
2. Preprocess images and labels.
3. Inject LoRA into U-Net.
4. Use Textual Prompt Embeddings.
5. Train the model.
6. Inference for conditioned image generation.


In [1]:
# Install necessary packages
!pip install pandas Pillow tqdm torch torchvision "transformers>=4.40.0" "diffusers>=0.27.0" scikit-learn IProgress ipywidgets "accelerate>=0.27.2" "peft>=0.10.0" matplotlib gradio safetensors

In [2]:
# Imports and Configuration
import os
import gc
import uuid
import json
import math
import time
from contextlib import nullcontext
import random
import numpy as np
import pandas as pd
import shutil
import gradio as gr
from PIL import Image as PILImage # Typed Annotation for Image
from tqdm import tqdm
from pathlib import Path
from typing import Tuple, Dict, List, Optional, Any
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
from torch.optim.lr_scheduler import CosineAnnealingLR, ReduceLROnPlateau
from torchvision import transforms

from diffusers import StableDiffusionPipeline, DDPMScheduler

from transformers import CLIPTokenizer

from peft import LoraConfig
from peft import get_peft_model_state_dict

In [3]:
# RUN_ID: str = str(uuid.uuid4()).replace('-', '')[:6]
# print(f"RUN_ID: {RUN_ID}")

# Define Batch Size
batch_size: int = 5

# Subset ratios for quick iteration (1.0 = full dataset)
train_subset_ratio: float = 1
valid_subset_ratio: float = 1
subset_seed: int = 42

# Dataset Paths
dataset_root: Path = Path('./datasets/appa-real-dataset_v2_improved')
labels_md_train = dataset_root / 'labels_metadata_train.csv'
labels_md_valid = dataset_root / 'labels_metadata_valid.csv'
labels_md_test  = dataset_root / 'labels_metadata_test.csv'

ds_train = dataset_root / 'train_data'
ds_valid = dataset_root / 'valid_data'
ds_test  = dataset_root / 'test_data'

# Load Metadata
df_md_train = pd.read_csv(labels_md_train)
df_md_valid = pd.read_csv(labels_md_valid)
df_md_test  = pd.read_csv(labels_md_test)

print(f"Train: {df_md_train.shape}, Valid: {df_md_valid.shape}, Test: {df_md_test.shape}")

# --- Optional: Rank by quality and cap per age bin using quality_report.csv ---
use_quality_rank: bool = True
quality_report_path: Path = dataset_root / 'reports' / 'quality_report.csv'


def apply_quality_cap_by_age(
    df: pd.DataFrame,
    report_path: Path,
    age_bin_size: int = 5,
    max_per_age_bin: int = 200,
    random_state: int = 42,
) -> pd.DataFrame:
    if not report_path.exists():
        print(f"[WARN] quality_report.csv not found at {report_path}. Skipping quality cap.")
        return df
    quality = pd.read_csv(report_path)
    if 'badness_score' not in quality.columns:
        print("[WARN] badness_score missing in quality report. Skipping quality cap.")
        return df
    quality = quality[['imageId', 'badness_score']].dropna()
    merged = df.merge(quality, on='imageId', how='left')
    merged['age_bin'] = (merged['age'] // age_bin_size) * age_bin_size
    out = []
    for age_bin, grp in merged.groupby('age_bin', sort=True):
        if len(grp) <= max_per_age_bin:
            out.append(grp)
            continue
        grp = grp.copy()
        # Lower badness_score = better quality; keep best N
        grp = grp.sort_values('badness_score', ascending=True)
        out.append(grp.head(max_per_age_bin))
    capped = pd.concat(out, ignore_index=True)
    capped = capped.drop(columns=['age_bin', 'badness_score'])
    return capped


# Fallback defaults if balancing params are not defined above
if 'age_bin_size' not in locals():
    age_bin_size = 5
if 'max_per_age_bin' not in locals():
    max_per_age_bin = 200

if use_quality_rank:
    df_md_train = apply_quality_cap_by_age(
        df_md_train,
        report_path=quality_report_path,
        age_bin_size=age_bin_size,
        max_per_age_bin=max_per_age_bin,
        random_state=subset_seed,
    )
    print(f"Quality-capped Train: {df_md_train.shape}")


Train: (2024, 5), Valid: (771, 5), Test: (940, 5)
Quality-capped Train: (1727, 5)


In [4]:
class ImageWithPromptDataset(Dataset):
    def __init__(
        self,
        df_md: pd.DataFrame,
        images_dir: Path,
        tokenizer: CLIPTokenizer,
        transform: Optional[transforms.Compose],
    ):
        self.df = df_md
        self.images_dir = images_dir
        self.tokenizer = tokenizer
        self.transform = transform

    def __len__(self) -> int:
        return len(self.df)

    def build_prompt(self,row):
        age_desc = f"{int(row['age'])} years old"
        gender_desc = row['gender']
        ethnicity_desc = row['ethnicity']

        # Emotion mapping
        emotion_map = {
            'neutral': "with a neutral expression",
            'happy': "smiling happily",
            'slightlyhappy': "smiling slightly",
            'other': "showing a subtle emotion"
        }
        emotion_desc = emotion_map.get(row['emotion'], "with an expression")  # fallback if unknown

        prompt = f"A {age_desc} {ethnicity_desc} {gender_desc} {emotion_desc}"
        return prompt

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        row = self.df.iloc[idx]
        img_name = f"{int(row['imageId']):06d}.jpg"
        img_path = self.images_dir / img_name

        image = PILImage.open(img_path).convert('RGB')
        if self.transform:
            image = self.transform(image)

        prompt = self.build_prompt(row)
        prompt_ids = self.tokenizer(prompt, return_tensors="pt", padding="max_length", truncation=True, max_length=77).input_ids[0]

        return {
            'pixel_values': image,
            'prompt_ids': prompt_ids
        }

In [5]:
# Data Transforms & Loaders
tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-large-patch14")

train_transform = transforms.Compose([
    transforms.Resize(512, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.RandomCrop(512),
    transforms.ToTensor(),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.05),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

val_transform = transforms.Compose([
    transforms.Resize(512, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.CenterCrop(512),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

train_dataset = ImageWithPromptDataset(df_md_train, ds_train, tokenizer, transform=train_transform)
valid_dataset = ImageWithPromptDataset(df_md_valid, ds_valid, tokenizer, transform=val_transform)

def _subset_dataset(dataset, ratio, seed):
    if ratio is None or ratio >= 1.0:
        return dataset
    ratio = max(0.0, min(1.0, ratio))
    n = len(dataset)
    k = max(1, int(n * ratio))
    rng = np.random.default_rng(seed)
    indices = rng.choice(n, size=k, replace=False).tolist()
    return Subset(dataset, indices)

train_dataset_for_loader = _subset_dataset(train_dataset, train_subset_ratio, subset_seed)
valid_dataset_for_loader = _subset_dataset(valid_dataset, valid_subset_ratio, subset_seed + 1)
train_dataset_for_sampling = train_dataset_for_loader

train_loader = DataLoader(train_dataset_for_loader, batch_size=batch_size, shuffle=True)
valid_loader = DataLoader(valid_dataset_for_loader, batch_size=batch_size)


In [6]:
# --------------------------------------------------------------------------------------------------
## Helper Functions
# --------------------------------------------------------------------------------------------------

def load_model(
    lora_r: int = 64,
    lora_alpha: int = 64,
    lora_dropout: float = 0.05,
    device: str = 'cuda',
    verbose: bool = False,
    use_mixed_precision: bool = True  # New parameter
) -> StableDiffusionPipeline:
    """
    Loads the Stable Diffusion pipeline with LoRA configuration and mixed precision support.
    
    Args:
        lora_r (int): LoRA rank.
        lora_alpha (int): LoRA alpha scaling.
        lora_dropout (float): Dropout probability for LoRA.
        device (str): Device to load the model onto.
        verbose (bool): If True, prints additional details during initialization.
        use_mixed_precision (bool): If True, uses FP16 for base model, FP32 for LoRA.
    
    Returns:
        StableDiffusionPipeline: The configured pipeline ready for training.
    """
    
    # 1. Load base pipeline in FP16 if mixed precision is enabled
    dtype = torch.float16 if use_mixed_precision else torch.float32
    pipe = StableDiffusionPipeline.from_pretrained(
        "runwayml/stable-diffusion-v1-5",
        torch_dtype=dtype,
        safety_checker=None,
        requires_safety_checker=False
    ).to(device)
    print(f"✅ Base model loaded in {dtype}.")
    
    # 2. Define LoRA PEFT config
    peft_lora_config = LoraConfig(
        r=lora_r,
        lora_alpha=lora_alpha,
        target_modules=["to_q", "to_v"],
        lora_dropout=lora_dropout,
        bias="none"
    )
    
    # 3. Add LoRA Adapter
    pipe.unet.add_adapter(adapter_name="age_gender_lora", adapter_config=peft_lora_config)
    print("✅ LoRA adapter added.")
    
    # 4. Enable LoRA layers for training
    pipe.unet.enable_lora()
    print("✅ LoRA adapter enabled for training.")
    
    # 5. Freeze VAE & Text Encoder
    pipe.vae.requires_grad_(False)
    pipe.text_encoder.requires_grad_(False)
    print("✅ VAE and Text Encoder frozen.")
    
    # 6. CRITICAL: Cast LoRA parameters to FP32 for stable training
    if use_mixed_precision:
        for name, param in pipe.unet.named_parameters():
            if 'lora' in name and param.requires_grad:
                param.data = param.data.to(torch.float32)
                if verbose:
                    print(f" - Cast {name} to FP32 for stable training")
    
    # Check if LoRA weights are non-zero (resumed run)
    is_resumed_run = False
    for name, module in pipe.unet.named_modules():
        if hasattr(module, 'lora_A') and hasattr(module.lora_A, 'weight'):
            if module.lora_A.weight.sum() != 0:
                is_resumed_run = True
                break
    
    # 7. Stabilize LoRA_B init for new runs (keep zero init)
    print("\n🔍 Stabilizing LoRA_B init for new runs...")
    for name, module in pipe.unet.named_modules():
        if hasattr(module, 'lora_B') and hasattr(module.lora_B, 'weight'):
            if not is_resumed_run:
                torch.nn.init.zeros_(module.lora_B.weight)
                if verbose:
                    print(f" - Zero-initialized lora_B for {name}")
    
    # 8. Debug and Verification
    print("\n🔍 Checking LoRA Layers Injected into UNet...")
    lora_param_count = 0
    for name, param in pipe.unet.named_parameters():
        if 'lora' in name:
            lora_param_count += 1
            param_dtype = param.dtype
            nan_inf = torch.isnan(param).any() or torch.isinf(param).any()
            if verbose: 
                print(f" - {name}: shape={param.shape}, dtype={param_dtype}, requires_grad={param.requires_grad}")
    
    if lora_param_count == 0:
        raise RuntimeError("❌ No LoRA parameters found in UNet! Injection failed.")
    
    # 9. Ensure ONLY LoRA Layers Are Trainable
    non_lora_trainable = [n for n, p in pipe.unet.named_parameters() if p.requires_grad and 'lora' not in n]
    if non_lora_trainable:
        print("❌ ERROR: Found non-LoRA parameters set to requires_grad=True:")
        for n in non_lora_trainable:
            print(f" - {n}")
        raise RuntimeError("Non-LoRA params are trainable! You must freeze them explicitly.")
    else:
        print("✅ Only LoRA layers are trainable.")
    
    print(f"\n🚀 LoRA setup successful. Base model in {dtype}, LoRA in FP32. Ready for training.")
    return pipe

def unload_model(pipe: StableDiffusionPipeline) -> None:
    """
    Unloads the UNet from the pipeline and clears GPU memory.

    Args:
        pipe (StableDiffusionPipeline): The current pipeline to unload from.
    """
    print("🔻 Unloading UNet to free GPU memory...")
    del pipe.unet
    torch.cuda.empty_cache()
    gc.collect()
    print("✅ UNet unloaded and GPU cache cleared.")

def setup_run_logging(
    output_base_dir: Path,
    resume_unet_checkpoint_path: Optional[Path] = None
) -> Dict[str, Any]:
    """
    Sets up the logging directory for a training run, generates a RUN_ID,
    and determines if the run is new or resumed.
    """
    is_resumed_run = False
    run_dir = None
    global generated_run_id
    
    if resume_unet_checkpoint_path and resume_unet_checkpoint_path.exists():
        is_resumed_run = True
        
        # --- 💡 Updated Logic Here ---
        # Traverse up the path to find the run directory name.
        # This assumes the structure is run_dir/lora_checkpoints/
        run_dir_name = resume_unet_checkpoint_path.parent.parent.name
        run_id = run_dir_name
        
        run_dir = output_base_dir / run_id
        if not run_dir.exists():
            print(f"Warning: Resuming from checkpoint but expected run directory {run_dir} not found. Creating it.")
            run_dir.mkdir(parents=True, exist_ok=True)
            
        print(f"🔄 Resuming existing run with RUN_ID: {run_id}")
        generated_run_id = run_id
    else:
        # New run
        run_id = f"{str(uuid.uuid4()).replace('-', '')[:6]}_run_{pd.Timestamp.now().strftime('%Y%m%d-%H%M%S')}"
        run_dir = output_base_dir / run_id
        run_dir.mkdir(parents=True, exist_ok=True)
        print(f"✨ Starting new run with RUN_ID: {run_id}")
        generated_run_id = run_id
    
    history_file = run_dir / "training_history.json"
    (run_dir / "lora_checkpoints").mkdir(exist_ok=True)
    (run_dir / "lora_samples").mkdir(exist_ok=True)
    
    return {
        "run_id": run_id,
        "run_dir": run_dir,
        "history_file": history_file,
        "is_resumed_run": is_resumed_run
    }

def load_or_resume_history(history_file: Path, is_resumed_run: bool) -> Dict[str, List[Dict[str, Any]]]:
    """Loads existing training history or initializes a new one."""
    training_history = {"epochs": []}
    if is_resumed_run and history_file.exists():
        with open(history_file, 'r') as f:
            training_history = json.load(f)
        print(f"📊 Loaded existing training history from {history_file}")
    return training_history



def save_run_config(run_dir: Path, config: Dict[str, Any], is_resumed_run: bool) -> Path:
    """Writes run config once per run for reproducibility."""
    config_path = run_dir / "run_config.json"
    if is_resumed_run and config_path.exists():
        return config_path
    with open(config_path, 'w') as f:
        json.dump(config, f, indent=4)
    return config_path

def save_best_checkpoints(
    pipe: StableDiffusionPipeline,
    output_dir: Path,
    current_epoch_number: int,
    current_loss: float,
    best_loss: float,
) -> Tuple[bool, Path, Path]:
    """
    Saves checkpoints only if the current loss is an improvement.
    Saves the TRUE LoRA adapter (small file) and the full U-Net for resuming.
    """
    output_dir.mkdir(parents=True, exist_ok=True)
    
    if current_loss >= best_loss:
        print(f"📉 Epoch {current_epoch_number}: Loss did not improve. Best loss remains {best_loss:.6f}.")
        return False, Path(""), Path("")

    print(f"🏆 Epoch {current_epoch_number}: Loss improved from {best_loss:.6f} to {current_loss:.6f}! Saving model...")

    # --- NEW: Define path for the TRUE LoRA adapter file ---
    # We will save it as a .safetensors file inside the directory.
    best_lora_adapter_dir = output_dir / "best_lora_adapter"
    best_lora_adapter_dir.mkdir(exist_ok=True)
    best_lora_adapter_path = best_lora_adapter_dir / "adapter_model.safetensors" # Correct filename
    
    best_full_unet_checkpoint_path = output_dir / "best_full_unet_checkpoint.pth"

    # Remove old checkpoints
    if best_lora_adapter_path.exists():
        best_lora_adapter_path.unlink()
        print("✅ Removed old best LoRA adapter file.")

    if best_full_unet_checkpoint_path.exists():
        best_full_unet_checkpoint_path.unlink()
        print("✅ Removed old best full U-Net checkpoint file.")

    # --- UPDATED SAVING LOGIC ---
    # 1. Save the TRUE LoRA adapter (will be small)
    lora_state_dict = get_peft_model_state_dict(pipe.unet, adapter_name="age_gender_lora")
    
    # Need to use the save_file utility from safetensors
    from safetensors.torch import save_file
    save_file(lora_state_dict, best_lora_adapter_path)
    
    # 2. Save the full U-Net state_dict for resuming training
    torch.save(pipe.unet.state_dict(), best_full_unet_checkpoint_path)
    
    print(f"Saved new best LoRA adapter (small) at {best_lora_adapter_path}")
    print(f"Saved new best full U-Net checkpoint (large) at {best_full_unet_checkpoint_path}")

    return True, best_lora_adapter_path, best_full_unet_checkpoint_path

def generate_and_save_sample(
    pipe: StableDiffusionPipeline,
    dataset: Dataset,
    sample_dir: Path,
    current_epoch_number: int,
    device: str
) -> Tuple[PILImage.Image, str, Path, Path]:
    """Generates a sample image, saves it with its prompt, and returns the results."""
    sample_dir.mkdir(parents=True, exist_ok=True)
    if hasattr(dataset, "df"):
        sample_row = random.choice(dataset.df.to_dict(orient="records"))
        sample_prompt: str = dataset.build_prompt(sample_row)
    else:
        base_dataset = dataset.dataset
        idx = random.choice(dataset.indices)
        sample_row = base_dataset.df.iloc[idx].to_dict()
        sample_prompt: str = base_dataset.build_prompt(sample_row)
    generator = torch.Generator(device=device).manual_seed(42)

    with torch.no_grad():
        with torch.amp.autocast(device):
            image = pipe(prompt=sample_prompt, num_inference_steps=150, guidance_scale=7.5, generator=generator).images[0]

    image_save_path = sample_dir / f"sample_epoch_{current_epoch_number}.png"
    image.save(image_save_path)
    print(f"Saved sample image at {image_save_path}")

    prompt_save_path = sample_dir / f"sample_epoch_{current_epoch_number}_prompt.txt"
    with open(prompt_save_path, 'w') as file:
        file.write(sample_prompt)
    print(f"Saved prompt to {prompt_save_path}")

    return image, sample_prompt, image_save_path, prompt_save_path

def display_image_with_prompt(image: PILImage.Image, prompt: str, current_epoch_number: int) -> None:
    """Displays a generated image along with its prompt using matplotlib."""
    plt.figure(figsize=(8, 8))
    plt.imshow(image)
    plt.title(f"Epoch {current_epoch_number} Sample\nPrompt: {prompt}", wrap=True)
    plt.axis('off')
    plt.show()

# --------------------------------------------------------------------------------------------------
## The Refactored `train_lora_diffusion` function
# --------------------------------------------------------------------------------------------------

def train_lora_diffusion(
    num_epochs: int,
    train_dataloader: DataLoader,
    valid_dataloader: DataLoader,
    dataset_for_sampling: Dataset,
    learning_rate: float = 2e-5,
    lora_r: int = 64,
    lora_alpha: int = 64,
    lora_dropout: float = 0.05,
    warmup_ratio: float = 0.1,
    min_lr: float = 1e-6,
    scheduler_type: str = "plateau",
    plateau_patience: int = 1,
    plateau_factor: float = 0.5,
    plateau_threshold: float = 1e-4,
    train_subset_ratio: float = 1.0,
    valid_subset_ratio: float = 1.0,
    subset_seed: Optional[int] = None,
    track_performance: bool = True,
    perf_log_interval_sec: float = 5.0,
    gradient_checkpoint_enable: bool = True,
    gradient_accumulation_steps: int = 1,
    output_base_dir: Path = Path("./lora_training_runs"),
    resume_unet_checkpoint_path: Optional[Path] = None,
    device: str = 'cuda',
    verbose: bool = False,
    use_gradio: bool = False,
    use_mixed_precision: bool = True  # New parameter
) -> Optional[gr.Blocks]:

    def _training_loop_generator():
        # Proactively clear cached VRAM between runs
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
        # Setup code remains the same...
        run_info = setup_run_logging(output_base_dir, resume_unet_checkpoint_path)
        run_dir, history_file, is_resumed_run = run_info["run_dir"], run_info["history_file"], run_info["is_resumed_run"]
        output_dir = run_dir / "lora_checkpoints"
        sample_dir = run_dir / "lora_samples"

        training_history = load_or_resume_history(history_file, is_resumed_run)
        run_config = {
            "num_epochs": num_epochs,
            "learning_rate": learning_rate,
            "lora_r": lora_r,
            "lora_alpha": lora_alpha,
            "lora_dropout": lora_dropout,
            "track_performance": track_performance,
            "perf_log_interval_sec": perf_log_interval_sec,
            "gradient_checkpoint_enable": gradient_checkpoint_enable,
            "gradient_accumulation_steps": gradient_accumulation_steps,
            "use_mixed_precision": use_mixed_precision,
            "optimizer": "AdamW",
            "scheduler": scheduler_type,
            "scheduler_min_lr": min_lr,
            "scheduler_patience": plateau_patience,
            "scheduler_factor": plateau_factor,
            "scheduler_threshold": plateau_threshold,
            "target_modules": ["to_q", "to_v"],
            "batch_size": getattr(train_dataloader, "batch_size", None),
            "train_dataset_size": len(train_dataloader.dataset) if train_dataloader is not None else None,
            "valid_dataset_size": len(valid_dataloader.dataset) if valid_dataloader is not None else None,
            "device": device,
            "perf_log_path": str(run_dir / "perf_log.jsonl")
        }
        if torch.cuda.is_available():
            props = torch.cuda.get_device_properties(0)
            run_config["cuda_name"] = props.name
            run_config["cuda_total_vram_gb"] = round(props.total_memory / (1024 ** 3), 3)
        train_images_dir = getattr(getattr(train_dataloader, "dataset", None), "images_dir", None)
        valid_images_dir = getattr(getattr(valid_dataloader, "dataset", None), "images_dir", None)
        if train_images_dir is not None:
            run_config["train_images_dir"] = str(train_images_dir)
        if valid_images_dir is not None:
            run_config["valid_images_dir"] = str(valid_images_dir)
        if "config" not in training_history:
            training_history["config"] = run_config
        config_path = save_run_config(run_dir, run_config, is_resumed_run)
        if use_gradio:
            yield {"status": f"✅ Saved run config to {config_path}"}
        else:
            print(f"✅ Saved run config to {config_path}")

        perf_log_path = run_dir / "perf_log.jsonl"
        last_perf_log_time = time.time()
        total_vram_gb = run_config.get("cuda_total_vram_gb")
        global_step = 0

        def _append_perf_log(payload: Dict[str, Any]) -> None:
            if not track_performance:
                return
            with open(perf_log_path, 'a') as f:
                f.write(json.dumps(payload) + "\n")

        start_epoch = len(training_history["epochs"])
        total_epochs = start_epoch + num_epochs
        steps_per_epoch = len(train_dataloader)
        updates_per_epoch = math.ceil(steps_per_epoch / gradient_accumulation_steps)
        total_update_steps = updates_per_epoch * total_epochs
        warmup_steps = int(total_update_steps * warmup_ratio)
        global_update_step = start_epoch * updates_per_epoch

        def _lr_for_step(step_idx: int) -> float:
            if total_update_steps <= 0:
                return learning_rate
            if warmup_steps > 0 and step_idx < warmup_steps:
                return learning_rate * (step_idx + 1) / max(1, warmup_steps)
            progress = (step_idx - warmup_steps) / max(1, total_update_steps - warmup_steps)
            cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
            return min_lr + (learning_rate - min_lr) * cosine

        if use_gradio:
            yield {"status": f"🚀 Training from epoch {start_epoch + 1} for a total of {total_epochs} epochs."}
        else:
            print(f"🚀 Training from epoch {start_epoch + 1} for a total of {total_epochs} epochs.")

        # Load model with mixed precision support
        pipe: StableDiffusionPipeline = load_model(
            lora_r, lora_alpha, lora_dropout, device, verbose, use_mixed_precision
        )

        try:
            pipe.safety_checker = lambda images, **kwargs: (images, [False] * len(images))

            noise_scheduler = DDPMScheduler.from_pretrained(
                "runwayml/stable-diffusion-v1-5",
                subfolder="scheduler"
            )

            if gradient_checkpoint_enable:
                pipe.unet.enable_gradient_checkpointing()
                status_msg = "✅ Gradient checkpointing enabled."
            else:
                status_msg = "✅ Gradient checkpointing is not enabled."
            if use_gradio:
                yield {"status": status_msg}
            else:
                print(status_msg)

            use_amp = use_mixed_precision and torch.cuda.is_available()

            def _autocast():
                if not use_amp:
                    return nullcontext()
                if hasattr(torch, "amp") and hasattr(torch.amp, "autocast"):
                    return torch.amp.autocast("cuda")
                return torch.cuda.amp.autocast()

            # Get trainable parameters (LoRA params in FP32)
            lora_params = [p for n, p in pipe.unet.named_parameters() if p.requires_grad]
            optimizer = torch.optim.AdamW(lora_params, lr=learning_rate)

            # Initialize GradScaler for mixed precision
            scaler = None
            if use_amp:
                if hasattr(torch, "amp") and hasattr(torch.amp, "GradScaler"):
                    scaler = torch.amp.GradScaler("cuda")
                else:
                    scaler = torch.cuda.amp.GradScaler()

            scheduler_type_norm = scheduler_type.lower().strip()
            if scheduler_type_norm in ("plateau", "reduceonplateau", "reducelronplateau"):
                scheduler = ReduceLROnPlateau(
                    optimizer,
                    mode="min",
                    factor=plateau_factor,
                    patience=plateau_patience,
                    threshold=plateau_threshold,
                    threshold_mode="rel",
                    min_lr=min_lr,
                )
            elif scheduler_type_norm in ("cosine", "cosineannealing", "cosineannealinglr"):
                scheduler = CosineAnnealingLR(optimizer, T_max=total_epochs, eta_min=min_lr)
            else:
                raise ValueError(f"Unknown scheduler_type: {scheduler_type}")

            if use_gradio:
                yield {"status": f"✅ Using {'mixed' if use_amp else 'full'} precision training."}
            else:
                print(f"✅ Using {'mixed' if use_amp else 'full'} precision training.")

            # Initialize best_loss
            best_loss = float('inf')
            if is_resumed_run and training_history['epochs']:
                if any(e.get('val_loss') is not None for e in training_history['epochs']):
                    best_loss = min(
                        e['val_loss'] for e in training_history['epochs'] if e.get('val_loss') is not None
                    )
                else:
                    best_loss = min(e['train_loss'] for e in training_history['epochs'])
                if use_gradio:
                    yield {"status": f"📊 Resuming from history. Initial best loss: {best_loss:.6f}"}
                else:
                    print(f"📊 Resuming from history. Initial best loss: {best_loss:.6f}")

            # Main Training Loop
            for epoch in range(start_epoch, total_epochs):
                pipe.unet.train()
                total_loss = 0.0
                processed_batches = 0
                current_epoch_number = epoch + 1
                epoch_start_time = time.perf_counter()
                epoch_step_time_sum = 0.0
                epoch_step_count = 0
                if track_performance and torch.cuda.is_available():
                    torch.cuda.reset_peak_memory_stats()

                grad_norm_values = []
                current_grad_norm = None

                tqdm_pbar = tqdm(train_dataloader, desc=f"Epoch {current_epoch_number}/{total_epochs}")
                for batch_idx, batch in enumerate(tqdm_pbar):
                    step_start_time = time.perf_counter()
                    # Move data to device with appropriate dtype
                    dtype = torch.float16 if use_amp else torch.float32
                    pixel_values = batch['pixel_values'].to(device, dtype=dtype)

                    # VAE encoding (no gradient needed)
                    with torch.no_grad():
                        latents = pipe.vae.encode(pixel_values).latent_dist.sample()
                        latents = latents * 0.18215

                    noise = torch.randn_like(latents)
                    timesteps = torch.randint(
                        0,
                        noise_scheduler.config.num_train_timesteps,
                        (latents.shape[0],),
                        device=device
                    ).long()
                    noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)
                    prompt_ids = batch['prompt_ids'].to(device)

                    # Text encoding (no gradient needed)
                    with torch.no_grad():
                        encoder_hidden_states = pipe.text_encoder(prompt_ids)[0]

                    # Forward pass with mixed precision
                    if use_amp:
                        with _autocast():
                            model_pred = pipe.unet(
                                noisy_latents,
                                timesteps,
                                encoder_hidden_states=encoder_hidden_states
                            ).sample
                            loss = nn.functional.mse_loss(model_pred, noise, reduction="mean")
                        loss_value = loss.item()
                        loss = loss / gradient_accumulation_steps

                        # Backward pass with gradient scaling
                        scaler.scale(loss).backward()
                    else:
                        # Full precision path
                        model_pred = pipe.unet(
                            noisy_latents,
                            timesteps,
                            encoder_hidden_states=encoder_hidden_states
                        ).sample
                        loss = nn.functional.mse_loss(model_pred, noise, reduction="mean")
                        loss_value = loss.item()
                        loss = loss / gradient_accumulation_steps
                        loss.backward()

                    total_loss += loss_value
                    processed_batches += 1

                    # Gradient accumulation and optimizer step
                    if (batch_idx + 1) % gradient_accumulation_steps == 0 or (batch_idx + 1) == len(train_dataloader):
                        current_lr = _lr_for_step(global_update_step)
                        for param_group in optimizer.param_groups:
                            param_group["lr"] = current_lr
                        if use_amp:
                            scaler.unscale_(optimizer)
                            grad_norm = torch.nn.utils.clip_grad_norm_(lora_params, max_norm=1.0)
                            scaler.step(optimizer)
                            scaler.update()
                        else:
                            grad_norm = torch.nn.utils.clip_grad_norm_(lora_params, max_norm=1.0)
                            optimizer.step()
                        if isinstance(grad_norm, torch.Tensor):
                            grad_norm = grad_norm.item()
                        current_grad_norm = grad_norm
                        grad_norm_values.append(grad_norm)
                        optimizer.zero_grad()
                        global_update_step += 1

                    step_time_sec = time.perf_counter() - step_start_time
                    epoch_step_time_sum += step_time_sec
                    epoch_step_count += 1
                    global_step += 1

                    if track_performance and (time.time() - last_perf_log_time) >= perf_log_interval_sec:
                        now_ts = time.time()
                        if torch.cuda.is_available():
                            allocated_gb = torch.cuda.memory_allocated() / (1024 ** 3)
                            reserved_gb = torch.cuda.memory_reserved() / (1024 ** 3)
                            max_allocated_gb = torch.cuda.max_memory_allocated() / (1024 ** 3)
                            max_reserved_gb = torch.cuda.max_memory_reserved() / (1024 ** 3)
                        else:
                            allocated_gb = None
                            reserved_gb = None
                            max_allocated_gb = None
                            max_reserved_gb = None
                        _append_perf_log({
                            "timestamp": now_ts,
                            "epoch": current_epoch_number,
                            "step": batch_idx + 1,
                            "global_step": global_step,
                            "step_time_sec": step_time_sec,
                            "loss": loss_value,
                            "avg_loss": (total_loss / processed_batches) if processed_batches > 0 else None,
                            "learning_rate": optimizer.param_groups[0]["lr"],
                            "grad_norm": current_grad_norm,
                            "allocated_vram_gb": allocated_gb,
                            "reserved_vram_gb": reserved_gb,
                            "max_allocated_vram_gb": max_allocated_gb,
                            "max_reserved_vram_gb": max_reserved_gb,
                            "total_vram_gb": total_vram_gb
                        })
                        last_perf_log_time = now_ts

                    # Update progress bar
                    avg_loss_running = total_loss / processed_batches if processed_batches > 0 else 0.0
                    tqdm_pbar.set_postfix({'loss': loss_value, 'avg_loss': avg_loss_running})

                # End of epoch logic
                avg_loss = total_loss / processed_batches if processed_batches > 0 else 0.0
                epoch_time_sec = time.perf_counter() - epoch_start_time
                avg_step_time_sec = (epoch_step_time_sum / epoch_step_count) if epoch_step_count > 0 else None
                steps_per_sec = (epoch_step_count / epoch_time_sec) if epoch_time_sec > 0 else None
                max_allocated_vram_gb = None
                max_reserved_vram_gb = None
                if track_performance and torch.cuda.is_available():
                    max_allocated_vram_gb = torch.cuda.max_memory_allocated() / (1024 ** 3)
                    max_reserved_gb = torch.cuda.max_memory_reserved() / (1024 ** 3)

                avg_grad_norm = None
                max_grad_norm = None
                min_grad_norm = None
                if grad_norm_values:
                    avg_grad_norm = sum(grad_norm_values) / len(grad_norm_values)
                    max_grad_norm = max(grad_norm_values)
                    min_grad_norm = min(grad_norm_values)

                avg_val_loss = None
                if valid_dataloader is not None:
                    pipe.unet.eval()
                    val_loss_total = 0.0
                    val_batches = 0
                    tqdm_val = tqdm(valid_dataloader, desc=f"Val {current_epoch_number}/{total_epochs}")
                    with torch.no_grad():
                        for val_batch in tqdm_val:
                            dtype = torch.float16 if use_amp else torch.float32
                            pixel_values = val_batch['pixel_values'].to(device, dtype=dtype)
                            latents = pipe.vae.encode(pixel_values).latent_dist.sample()
                            latents = latents * 0.18215
                            noise = torch.randn_like(latents)
                            timesteps = torch.randint(
                                0,
                                noise_scheduler.config.num_train_timesteps,
                                (latents.shape[0],),
                                device=device
                            ).long()
                            noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)
                            prompt_ids = val_batch['prompt_ids'].to(device)
                            encoder_hidden_states = pipe.text_encoder(prompt_ids)[0]
                            if use_amp:
                                with _autocast():
                                    model_pred = pipe.unet(
                                        noisy_latents,
                                        timesteps,
                                        encoder_hidden_states=encoder_hidden_states
                                    ).sample
                                    val_loss = nn.functional.mse_loss(model_pred, noise, reduction="mean")
                            else:
                                model_pred = pipe.unet(
                                    noisy_latents,
                                    timesteps,
                                    encoder_hidden_states=encoder_hidden_states
                                ).sample
                                val_loss = nn.functional.mse_loss(model_pred, noise, reduction="mean")
                            val_loss_total += val_loss.item()
                            val_batches += 1
                            tqdm_val.set_postfix({'val_loss': val_loss.item()})
                    avg_val_loss = val_loss_total / val_batches if val_batches > 0 else None
                val_loss_str = f"{avg_val_loss:.6f}" if avg_val_loss is not None else "n/a"
                metric_loss = avg_val_loss if avg_val_loss is not None else avg_loss

                if scheduler is not None:
                    if scheduler_type_norm in ("plateau", "reduceonplateau", "reducelronplateau"):
                        scheduler.step(metric_loss)
                    else:
                        scheduler.step()

                current_lr = optimizer.param_groups[0]["lr"]

                is_saved, lora_path, unet_path = save_best_checkpoints(
                    pipe, output_dir, current_epoch_number, metric_loss, best_loss
                )

                if is_saved:
                    best_loss = metric_loss

                image, prompt, image_path, prompt_path = generate_and_save_sample(
                    pipe, dataset_for_sampling, sample_dir, current_epoch_number, device
                )

                epoch_history = {
                    "epoch": current_epoch_number,
                    "train_loss": avg_loss,
                    "val_loss": avg_val_loss,
                    "best_loss": best_loss,
                    "saved_this_epoch": is_saved,
                    "sample_prompt": prompt,
                    "sample_image_path": str(image_path),
                    "lora_adapter_path": str(lora_path) if is_saved else None,
                    "full_unet_checkpoint_path": str(unet_path) if is_saved else None,
                    "learning_rate": current_lr,
                    "avg_grad_norm": avg_grad_norm,
                    "max_grad_norm": max_grad_norm,
                    "min_grad_norm": min_grad_norm,
                    "last_grad_norm": current_grad_norm,
                    "epoch_time_sec": epoch_time_sec,
                    "avg_step_time_sec": avg_step_time_sec,
                    "steps_per_sec": steps_per_sec,
                    "max_allocated_vram_gb": max_allocated_vram_gb,
                    "max_reserved_vram_gb": max_reserved_gb,
                    "total_vram_gb": total_vram_gb
                }
                training_history["epochs"].append(epoch_history)

                with open(history_file, 'w') as f:
                    json.dump(training_history, f, indent=4)

                if use_gradio:
                    yield {
                        "epoch": current_epoch_number,
                        "train_loss": avg_loss,
                        "val_loss": avg_val_loss,
                        "learning_rate": current_lr,
                        "avg_grad_norm": avg_grad_norm,
                        "max_grad_norm": max_grad_norm,
                        "min_grad_norm": min_grad_norm,
                        "last_grad_norm": current_grad_norm,
                        "sample_image": image,
                        "sample_prompt": prompt,
                        "history_log": epoch_history,
                        "status": f"✅ Epoch {current_epoch_number}/{total_epochs} completed. Train: {avg_loss:.6f}, Val: {val_loss_str}. Best: {best_loss:.6f}"
                    }
                else:
                    print(f"Epoch [{current_epoch_number}/{total_epochs}] Train Loss: {avg_loss:.6f}, Val Loss: {val_loss_str}, Learning Rate: {current_lr:.6e}")
                    print(f"Best Loss so far: {best_loss:.6f}")
                    print(f"Updated training history saved to {history_file}")
                    display_image_with_prompt(image, prompt, current_epoch_number)

        finally:
            unload_model(pipe)

        if use_gradio:
            yield {"status": "✨ Training complete! Model unloaded."}
        else:
            print("✨ Training complete! Model unloaded.")

    # --- Conditional Execution based on `use_gradio` flag ---
    if use_gradio:
        with gr.Blocks(title="LoRA Training Dashboard") as demo:
            gr.Markdown("# LoRA Training Dashboard 🚀")
            with gr.Row():
                with gr.Column(scale=1):
                    with gr.Accordion("Training Settings", open=True):
                        num_epochs_input = gr.Slider(label="Number of Epochs", minimum=1, maximum=100, value=num_epochs, step=1)
                        lr_input = gr.Number(label="Learning Rate", value=learning_rate)
                        lora_r_input = gr.Slider(label="LoRA Rank (r)", minimum=1, maximum=256, value=lora_r, step=1)
                        lora_alpha_input = gr.Slider(label="LoRA Alpha", minimum=1, maximum=256, value=lora_alpha, step=1)
                        lora_dropout_input = gr.Number(label="LoRA Dropout", value=lora_dropout)
                        grad_accum_input = gr.Slider(label="Gradient Accumulation Steps", minimum=1, maximum=16, value=gradient_accumulation_steps, step=1)
                    start_button = gr.Button("Start Training", variant="primary")
                    status_box = gr.Textbox(label="Status", lines=2)
                    history_box = gr.JSON(label="Last Epoch Details")
                with gr.Column(scale=2):
                    gr.Markdown("### Training Loss")
                    loss_plot = gr.Plot(value=pd.DataFrame({"Epoch": [], "Loss": []}))
                    gr.Markdown("### Sample Generation")
                    with gr.Row():
                        sample_image_output = gr.Image(label="Generated Sample", width=512)
                        sample_prompt_output = gr.Textbox(label="Prompt", lines=5)

            training_history_df = {"Epoch": [], "Loss": []}
            def update_gradio_plot_and_status(generator_outputs):
                nonlocal training_history_df
                if "train_loss" in generator_outputs:
                    training_history_df["Epoch"].append(generator_outputs["epoch"])
                    training_history_df["Loss"].append(generator_outputs["train_loss"])
                    df = pd.DataFrame(training_history_df)
                    return {
                        loss_plot: gr.update(value=df, x="Epoch", y="Loss", title="Training Loss over Epochs"),
                        sample_image_output: gr.update(value=generator_outputs["sample_image"]),
                        sample_prompt_output: gr.update(value=generator_outputs["sample_prompt"]),
                        status_box: gr.update(value=generator_outputs["status"]),
                        history_box: gr.update(value=generator_outputs["history_log"])
                    }
                else:
                    return {
                        status_box: gr.update(value=generator_outputs["status"]),
                    }

            start_button.click(
                fn=_training_loop_generator,
                inputs=[
                    num_epochs_input, lr_input, lora_r_input, lora_alpha_input, lora_dropout_input, grad_accum_input
                ],
                outputs=[loss_plot, sample_image_output, sample_prompt_output, status_box, history_box],
            )

        return demo
    else:
        for _ in _training_loop_generator():
            pass
        return None


In [7]:
# Random Sample of ImageDataSet Prompt
dataset = train_loader.dataset
if hasattr(dataset, "df"):
    sample_row = random.choice(dataset.df.to_dict(orient="records"))
    sample_prompt = dataset.build_prompt(sample_row)
else:
    base_dataset = dataset.dataset
    idx = random.choice(dataset.indices)
    sample_row = base_dataset.df.iloc[idx].to_dict()
    sample_prompt = base_dataset.build_prompt(sample_row)
print(sample_prompt)

A 15 years old caucasian female smiling happily


In [ ]:
# --- To train from scratch ---
demo = train_lora_diffusion(
    num_epochs=25,
    train_dataloader=train_loader,
    valid_dataloader=valid_loader,
    dataset_for_sampling=train_dataset_for_sampling,
    learning_rate=2e-5,
    scheduler_type="plateau",
    plateau_patience=2,
    plateau_factor=0.3,
    plateau_threshold=1e-4,
    min_lr=1e-6,
    lora_r=64,
    lora_alpha=64,
    lora_dropout=0.05,
    train_subset_ratio=train_subset_ratio,
    valid_subset_ratio=valid_subset_ratio,
    subset_seed=subset_seed,
    gradient_checkpoint_enable=True,
    gradient_accumulation_steps=4, # Example: accumulate gradients over 4 steps
    output_base_dir=Path("./lora_training_runs"), # Change 'output_dir' to 'output_base_dir'
    # The 'sample_dir' parameter also needs to be removed from the call,
    # as it's now internally derived from 'output_base_dir' and 'run_dir'.
    resume_unet_checkpoint_path=None, # Set to None for training from scratch
    verbose=False,
    use_gradio=False,  # <-- Enable Gradio here
    use_mixed_precision=True  # Enable mixed precision
)

if demo:
    demo.launch(share=False) # Create public URL for Google Colab monitoring

✨ Starting new run with RUN_ID: 8436e6_run_20260129-162112
✅ Saved run config to lora_training_runs/8436e6_run_20260129-162112/run_config.json
🚀 Training from epoch 1 for a total of 25 epochs.


Loading pipeline components...:   0%|          | 0/6 [00:00<?, ?it/s]

`torch_dtype` is deprecated! Use `dtype` instead!


✅ Base model loaded in torch.float16.
✅ LoRA adapter added.
✅ LoRA adapter enabled for training.
✅ VAE and Text Encoder frozen.

🔍 Stabilizing LoRA_B init for new runs...

🔍 Checking LoRA Layers Injected into UNet...
✅ Only LoRA layers are trainable.

🚀 LoRA setup successful. Base model in torch.float16, LoRA in FP32. Ready for training.
✅ Gradient checkpointing enabled.
✅ Using mixed precision training.


Epoch 1/25:   0%|          | 1/346 [00:02<11:55,  2.07s/it, loss=0.0986, avg_loss=0.0986]

In [ ]:
# ######################################
# ##### SET RUN_ID TO RESUME FROM ######
# ######################################
# # 💡 ROBUST CHECK: Check if 'generated_run_id' exists from a previous run
# if 'generated_run_id' in globals():
#     RESUME_RUN_ID = generated_run_id
#     print(f"🔄 Using dynamically generated RUN_ID: {RESUME_RUN_ID}")
# else:
#     # Fallback to a placeholder if the 'train from scratch' cell hasn't been run
#     RESUME_RUN_ID = '40343f_run_20260125-162752'
#     print(f"⚠️ Using placeholder RUN_ID: {RESUME_RUN_ID}. Please update this if resuming.")

# RESUME_EPOCH = 5

# # --- To resume training ---
# # Assuming you have a checkpoint saved from a previous run, e.g., unet_lora_weights_epoch_5.pth
# demo = train_lora_diffusion(
#     num_epochs=25, # Train for 5 more epochs
#     train_dataloader=train_loader,
#     valid_dataloader=valid_loader,
#     dataset_for_sampling=train_dataset_for_sampling,
#     learning_rate=2e-5, # Lower LR for resumed training
#     lora_r=64,
#     lora_alpha=64,
#     lora_dropout=0.05,
# #     train_subset_ratio=train_subset_ratio,
#     valid_subset_ratio=valid_subset_ratio,
#     subset_seed=subset_seed,
#     gradient_checkpoint_enable=True,
#     gradient_accumulation_steps=4, # Example: accumulate gradients over 4 steps
#     output_base_dir=Path("./lora_training_runs"), # Change 'output_dir' to 'output_base_dir'
#     resume_unet_checkpoint_path=Path(f"./lora_training_runs/{RESUME_RUN_ID}/lora_checkpoints/full_unet_checkpoint_epoch_{RESUME_EPOCH}.pth"), # Update path to reflect new structure
#     verbose=False,
#     use_gradio=False,  # <-- Enable Gradio here
#     use_mixed_precision=True  # Enable mixed precision
# )

# if demo:
#     demo.launch(share=False) # Create public URL for Google Colab monitoring